In [1]:
#import relevant libraries
import os
#from scipy import stats

import numpy as np
#import scipy as sp
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm
import seaborn as sns
import dabest
import NLCLIMB
import NLMATH
import itertools
from datetime import datetime
date = datetime.today().strftime('%Y%m%d')
from statistics import mean
from textwrap import wrap


import dabest
import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly.graph_objects import Layout
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

#NOTE: SUPPRESSES WARNINGS!

import warnings


warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)


Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:00<00:00, 22.84it/s]


Numba compilation complete!


In [3]:
#initial file processing
workcomp = "C:\\Users\\User"
computer2 = "C:\\Users\\lnico"
officecomp = "C:\\Users\\Star"
homecomp = "D:"
titledpath = homecomp
filenumberfolder = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\MBON thinks\\numbers\\"

#OSAR file names to change if there are updates. No filename to change for Falling as it is Automatic
fileACR = "20250523_ACR_totalcompilation.csv"
fileCR2 = "20250523_Chrimson2_totalcompilation.csv"

In [4]:
def wrap_labels(ax, width, break_long_words=False):
    import textwrap
    labels = []
    for label in ax.get_xticklabels():
        text = label.get_text()
        labels.append(textwrap.fill(text, width=width,
                      break_long_words=break_long_words))
    ax.set_xticklabels(labels, rotation=0)

## Falling counting

In [5]:
def countingnumbers(fileread):
    dffileread = pd.read_csv(fileread)
    flynumbers = len(dffileread.columns)-3
    
    return int(flynumbers*0.5)


In [6]:
filenumberdir = filenumberfolder + "MBON_numbers.csv"
filefalling = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\Falling_New\\"
openPath = titledpath + filefalling

numberdf = pd.read_csv(titledpath +filenumberdir, encoding='latin-1')

numberdf2 = numberdf.copy()

for n in numberdf2['Name']:
    dffallingtypes = [i for i in os.listdir(openPath) if n in i]
    for y in dffallingtypes:
        if y.endswith("ACR.csv") and y.startswith(n):
            x = countingnumbers(openPath + y)
            numberdf2.loc[(numberdf2['Name'] == n), ["F_ACR"]] = x
        elif y.endswith("ACR.csv") and y.startswith("w1118"):
            x = countingnumbers(openPath + y)
            numberdf2.loc[(numberdf2['Name'] == n), ["F_ACR_wt"]] = x
        elif y.endswith("Chrimson2.csv") and y.startswith(n):
            x = countingnumbers(openPath + y)
            numberdf2.loc[(numberdf2['Name'] == n), ["F_Chrimson2"]] = x
        elif y.endswith("Chrimson2.csv") and y.startswith("w1118"):
            x = countingnumbers(openPath + y)  
            numberdf2.loc[(numberdf2['Name'] == n), ["F_Chrimson2_wt"]] = x

## OSAR counting

In [7]:
fileosar = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\osar_compiled\\"
totalosarfiles = titledpath + fileosar

filechoice_ACR = totalosarfiles + fileACR
readfilechoice_ACR = pd.read_csv(filechoice_ACR)
filechoice_CR2 = totalosarfiles + fileCR2
readfilechoice_CR2 = pd.read_csv(filechoice_CR2)
parsedfilechoice_ACR = readfilechoice_ACR.drop(readfilechoice_ACR.columns[[0]],axis = 1)
parsedfilechoice_CR2 = readfilechoice_CR2.drop(readfilechoice_CR2.columns[[0]],axis = 1)
totalparsedchoice = pd.concat([parsedfilechoice_ACR, parsedfilechoice_CR2], axis = 0).reset_index(drop=True)

fulllightchoice = totalparsedchoice[totalparsedchoice["light_intensity"] == "Full"]



In [8]:
numberdf3 = numberdf2.copy()
for n in numberdf3['Name']:
    specificmbonfull = fulllightchoice[fulllightchoice["driver"] == n]
    numberdf3.loc[(numberdf3['Name'] == n), ["O_ACR"]] = len(specificmbonfull[(specificmbonfull['status'] == 'Offspring') & (specificmbonfull['opsin'] == 'GtACR1')])
    numberdf3.loc[(numberdf3['Name'] == n), ["O_ACR_wt"]] = len(specificmbonfull[(specificmbonfull['status'] == 'Sibling') & (specificmbonfull['opsin'] == 'GtACR1')])
    numberdf3.loc[(numberdf3['Name'] == n), ["O_Chrimson2"]] = len(specificmbonfull[(specificmbonfull['status'] == 'Offspring') & (specificmbonfull['opsin'] == 'Chrimson')])
    numberdf3.loc[(numberdf3['Name'] == n), ["O_Chrimson2_wt"]] = len(specificmbonfull[(specificmbonfull['status'] == 'Sibling') & (specificmbonfull['opsin'] == 'Chrimson')])


## File compilation

In [9]:
numberdf4 = numberdf3.replace(0, '')
numberdf4.to_csv(titledpath + filenumberfolder + date + ' MBON_numbers.csv', index=False)  

In [10]:
import plotly.graph_objects as go
import textwrap

numberdf5 = numberdf4.drop(columns=['Genotype'])
numberdf5 = numberdf5.replace(np.nan, "")

# Wrap headers at underscores
wrapped_headers = [col.replace('Chrimson2', 'Chrimson').replace('_', '<br>') for col in numberdf5.columns]

# Apply wrapping and alignment to each cell
cell_values = [
    [f"{val:.0f}".rjust(8) if isinstance(val, (int, float)) else str(val).rjust(8) for val in numberdf5[col]]
    for col in numberdf5.columns
]

# Conditional cell background colors
cell_fill_colors = []
for col in numberdf5.columns:
    col_colors = []
    for val in numberdf5[col]:
        try:
            # Check if it's a relevant numeric column
            if any(col.startswith(prefix) for prefix in ["F_", "O_"]) and ("ACR" in col or "Chrimson" in col):
                threshold = 90 if col.endswith("_wt") else 45
                if float(val) < threshold:
                    col_colors.append("yellow")
                else:
                    col_colors.append("rgba(0,0,0,0)")  # transparent
            else:
                col_colors.append("rgba(0,0,0,0)")
        except:
            col_colors.append("rgba(0,0,0,0)")
    cell_fill_colors.append(col_colors)
    
# Define figure
fig = go.Figure(data=[go.Table(
    columnwidth=[1]*len(numberdf5.columns),  # Uniform column width
    header=dict(
        values=[f"<b>{h.rjust(8)}</b>" for h in wrapped_headers],
        fill_color="#7d7f7c",  # Shaded header
        line_color="black",    # Border color
        align="right",
        font=dict(color="black", size=12),
        height=40
    ),
    cells=dict(
        values=cell_values,
        fill_color=cell_fill_colors,  
        line_color="black",         # Border color for cells
        align="right",
        font=dict(color="black", size=11),
        height=30
    )
)])

# Set figure size (width x height in pixels)
fig.update_layout(
    width=1000,
    height=1720,
    margin=dict(l=0, r=0, t=0, b=0)
)

# Show figure in notebook or interactive environment
fig.show()
fig.write_image(titledpath + filenumberfolder + date + "table_output.svg")